In [16]:
from os.path import realpath

# Local steps to run in pycharm
# conda activate sage
# jupyter lab --no-browser --port=8888
# configure external server with the token
# sagemath (10.7)

from sage.all import *
from sage.plot.plot import graphics_array
from itertools import combinations
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})

def graph_square(H):
    G = Graph()
    G.add_vertices(H.vertices())
    for u in H.vertices():
        for v in H.vertices():
            if u < v and H.distance(u,v) <= 2:
                G.add_edge(u,v)
    return G

def is_square_root(G, H):
    return graph_square(H) == G

def find_square_root(G):
    V = list(G.vertices())
    n = len(V)
    pairs = list(combinations(range(n), 2))

    total = 1 << len(pairs)
    for mask in range(total):
        H = Graph()
        H.add_vertices(V)
        for bit, (i, j) in enumerate(pairs):
            if (mask >> bit) & 1:
                H.add_edge(V[i], V[j])
        if is_square_root(G, H):
            return H
    return None

In [17]:
# Efficient find_square_root function

def violates_partial(G, H):
    for u, v in combinations(H.vertices(), 2):
        try:
            d = H.distance(u, v)
        except (ValueError, RuntimeError):
            continue

        if H.has_edge(u, v) and not G.has_edge(u, v):
            return True
        if d <= 2 and not G.has_edge(u, v):
            return True
    return False

def find_square_root(G):
    V = list(G.vertices())
    pairs = list(combinations(V, 2))
    H = Graph()
    H.add_vertices(V)

    def backtrack(idx):
        if idx == len(pairs):
            return H.copy() if is_square_root(G, H) else None

        u, v = pairs[idx]

        H.add_edge(u, v)
        if not violates_partial(G, H):
            result = backtrack(idx + 1)
            if result:
                return result
        H.delete_edge(u, v)

        if not violates_partial(G, H):
            result = backtrack(idx + 1)
            if result:
                return result

        return None

    return backtrack(0)

In [18]:
# LP with max objective function
def max_linear_program(G):
    m = MixedIntegerLinearProgram(solver='Gurobi', maximization=False)
    m.solver_parameter("Method", 2)

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    vars = {
        'x': {},
        'y': {},
        'z': {}
    }

    for i in range(n):
        for j in range(n):
            vars['x'][(i,j)] = x[i,j]
            vars['z'][(i,j)] = z[i,j]
            for k in range(n):
                vars['y'][(i,k,j)] = y[i,k,j]

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    m.set_objective(- m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, vars

# Dual of max LP
def dual_linear_program(G):
    m = MixedIntegerLinearProgram(solver='Gurobi', maximization=True)
    m.solver_parameter("Method", 2)

    alpha   = m.new_variable(real=True)
    beta    = m.new_variable(real=True)
    gamma   = m.new_variable(real=True, nonnegative=True)
    delta   = m.new_variable(real=True, nonnegative=True)
    epsilon = m.new_variable(real=True, nonnegative=True)
    zeta    = m.new_variable(real=True, nonnegative=True)
    eta     = m.new_variable(real=True, nonnegative=True)
    theta   = m.new_variable(real=True)
    iota    = m.new_variable(real=True)
    lambda_ = m.new_variable(real=True, nonnegative=True)
    mu      = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    vars = {
        'alpha': {},
        'beta': {},
        'gamma': {},
        'delta': {},
        'epsilon': {},
        'zeta': {},
        'eta': {},
        'theta': {},
        'iota': {},
        'lambda': {},
        'mu': {}
    }

    for i in range(n):
        vars['alpha'][i] = alpha[i]
        vars['iota'][i] = iota[i]

        for j in range(n):
            vars['beta'][(i,j)] = beta[i,j]
            vars['gamma'][(i,j)] = gamma[i,j]
            vars['delta'][(i,j)] = delta[i,j]
            vars['theta'][(i,j)] = theta[i,j]
            vars['lambda'][(i,j)] = lambda_[i,j]

            for k in range(n):
                vars['epsilon'][(i,k,j)] = epsilon[i,k,j]
                vars['zeta'][(i,k,j)] = zeta[i,k,j]
                vars['eta'][(i,k,j)] = eta[i,k,j]
                vars['mu'][(i,k,j)] = mu[i,k,j]

    # x_ij dual constraints
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(
                    -1
                    - sum(epsilon[i,i,k] for k in range(n))
                    - sum(zeta[k,i,i] for k in range(n))
                    + sum(eta[i,i,k] for k in range(n))
                    + sum(eta[k,i,i] for k in range(n))
                    + iota[i]
                    + lambda_[i,i]
                    >= 0
                )
            else:
                m.add_constraint(
                    -1
                    + alpha[i]
                    - beta[i,j]
                    - sum(epsilon[i,j,k] for k in range(n))
                    - sum(zeta[k,i,j] for k in range(n))
                    + sum(eta[i,j,k] for k in range(n))
                    + sum(eta[k,i,j] for k in range(n))
                    + theta[i,j]
                    - theta[j,i]
                    + lambda_[i,j]
                    >= 0
                )

    # y_ikj dual constraints
    for i in range(n):
        for k in range(n):
            for j in range(n):
                if j == i:
                    m.add_constraint(
                        - alpha[i]
                        + epsilon[i,k,i]
                        + zeta[i,k,i]
                        - eta[i,k,i]
                        + mu[i,k,i]
                        >= 0
                    )
                else:
                    m.add_constraint(
                        - beta[i,j]
                        + epsilon[i,k,j]
                        + zeta[i,k,j]
                        - eta[i,k,j]
                        + mu[i,k,j]
                        >= 0
                    )

    # z_ij dual constraints
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(
                    alpha[i]
                    + gamma[i,i]
                    - delta[i,i]
                    == 0
                )
            else:
                m.add_constraint(
                    beta[i,j]
                    + gamma[i,j]
                    - delta[i,j]
                    == 0
                )

    m.set_objective(
        sum(
            -M * A_G[i,j] * gamma[i,j]
            + A_G[i,j] * delta[i,j]
            - lambda_[i,j]
            for i in range(n) for j in range(n)
        )
        - sum(
            eta[i,k,j] + mu[i,k,j]
            for i in range(n)
            for k in range(n)
            for j in range(n)
        )
    )

    return m, vars

# Solve both primal and dual and get the gap
def solve_primal_dual(G):
    mp, primal_vars = max_linear_program(G)
    try:
        mp.solve()
        primal_val = mp.get_objective_value()
        primal_status = "solved"

        primal_solution = {}
        for name, block in primal_vars.items():
            values = {}
            for key, var in block.items():
                values[key] = mp.get_values(var)
            primal_solution[name] = values
    except MIPSolverException as e:
        primal_val = None
        primal_status = "failed"
        primal_solution = None

    md, dual_vars = dual_linear_program(G)
    try:
        md.solve()
        dual_val = md.get_objective_value()
        dual_status = "solved"

        dual_solution = {}
        for name, block in dual_vars.items():
            values = {}
            for key, var in block.items():
                values[key] = md.get_values(var)
            dual_solution[name] = values
    except MIPSolverException as e:
        dual_val = None
        dual_status = "failed"
        dual_solution = None

    if primal_val is not None and dual_val is not None:
        gap = dual_val - primal_val
    else:
        gap = None

    return {
        "primal_val": primal_val,
        "dual_val": dual_val,
        "gap": gap,
        "primal_status": primal_status,
        "dual_status": dual_status,
    }, primal_solution, dual_solution

In [19]:
# Generate all possible graph squares/roots for given n
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sage.graphs.graph_generators import graphs as graph_gens
from sage.numerical.mip import MIPSolverException

n = 7
ncpus = 12
dir = f"dual_graph_output"
output_dir = os.path.join(dir, f"n={n}")
output_pdf = os.path.join(output_dir, f"graph_results_n={n}.pdf")
comparison_pdf = os.path.join(output_dir, f"graph_comparison_n={n}.pdf")
summary_file = os.path.join(output_dir, f"graph_summary_n={n}.txt")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(dir, exist_ok=True)

def error_plot(message):
    g = Graphics()
    g += text(message, (0, 0), fontsize=14, color="red", horizontal_alignment='center')
    g.axes(False)
    return g

def clean_number(x, digits=3, tol=1e-10):
    if x is None:
        return None
    if abs(x) < tol:
        return 0.0
    if abs(x - round(x)) < tol:
        return float(round(x))
    return round(x, digits)

def fmt(x, width=8, digits=3):
    if x is None:
        return f"{'NA':>{width}}"
    return f"{x:>{width}.{digits}f}"

def process_graph(idx, G):
    G.relabel(range(G.order()))
    n = G.order()

    A_H = None
    sol_H = None
    H = None
    lp_status = "unknown"
    has_root = False
    edges_H = None
    is_noninteger = False

    lp_results, primal_solution, dual_solution = solve_primal_dual(G)

    primal_val = clean_number(lp_results["primal_val"])
    dual_val   = clean_number(lp_results["dual_val"])
    gap        = clean_number(lp_results["gap"])

    try:
        m, vars = max_linear_program(G)
        m.solve()

        solution = {}
        for (i, j), var in vars['x'].items():
            solution[(i, j)] = m.get_values(var)

        A_H = matrix([
            [solution[i, j] for j in range(n)]
            for i in range(n)
        ])

        sol_H = Graph(A_H, weighted=True)
        for (u, v, w) in sol_H.edges():
            sol_H.set_edge_label(u, v, round(w, 3))

        p1 = sol_H.plot(edge_labels=True, title='Weighted LP H')
        lp_status = "feasible"

        if any(abs(A_H[i, j] - round(A_H[i, j])) > 1e-6 for i in range(n) for j in range(n)):
            is_noninteger = True

    except MIPSolverException:
        sol_H = None
        p1 = error_plot("LP infeasible")
        lp_status = "infeasible"

    H = find_square_root(G)

    if H is None:
        p2 = error_plot("No feasible H")
        has_root = False
        edges_H = None
    else:
        p2 = H.plot(title='Feasible H')
        has_root = True
        edges_H = H.size()


    p3 = G.plot(title='Original G')
    combined = graphics_array([p1, p2, p3])

    summary = {
        "index": idx,
        "edges_G": G.size(),
        "vertices_G": n,
        "lp_status": lp_status,
        "has_root": has_root,
        "edges_H": edges_H,
        "is_noninteger": is_noninteger,
        "primal": primal_val,
        "dual": dual_val,
        "gap": gap
    }

    summary["weak_duality_violation"] = gap is not None and gap < -1e-6
    summary["strong_duality_violation"] = gap is not None and abs(gap) > 1e-6

    is_mismatch = (is_noninteger or (lp_status == "feasible" and H is None))

    if is_mismatch:
        if sol_H is not None:
            p_left = sol_H.plot(edge_labels=True, title="Weighted LP H")
        else:
            p_left = error_plot("No LP solution")

        if H is not None:
            p_mid = H.plot(title="Feasible H")
        else:
            p_mid = error_plot("No discrete H")

        p_right = G.plot(title="Original G")
        comparison = graphics_array([p_left, p_mid, p_right])
    else:
        comparison = None

    return combined, summary, comparison, is_mismatch, primal_solution, dual_solution

def run_task(idx, G):
    return process_graph(idx, G)

tasks = [(i, Graph(G0)) for i, G0 in enumerate(graph_gens.nauty_geng(str(n)))]

results = []
mismatch_results = []

def one_line_solution_summary(sol, tol=1e-8):
    if sol is None:
        return "None"

    block_parts = []
    total = 0

    for name, block in sol.items():
        nz = sum(1 for v in block.values() if abs(v) > tol)
        if nz > 0:
            block_parts.append(f"{name}:{nz}")
            total += nz

    if total == 0:
        return "All0"

    return f"{total}nz [" + ", ".join(block_parts) + "]"

for idx, G in tasks:
    plot, summary, comparison, is_mismatch, primal_solution, dual_solution = run_task(idx, G)
    results.append((plot, summary))
    if is_mismatch and comparison is not None:
        mismatch_results.append((comparison, summary))

    primal_line = one_line_solution_summary(primal_solution)
    dual_line   = one_line_solution_summary(dual_solution)

    print(
        f"✅ Graph {summary['index']:>4} | "
        f"LP={summary['lp_status']:<10} | "
        f"Root={str(summary['has_root']):<5} | "
        f"Primal={fmt(summary['primal'])} | "
        f"Dual={fmt(summary['dual'])} | "
        f"Gap={fmt(summary['gap'])}"
        f"{'  ⚠️ mismatch' if is_mismatch else ''}"
    )
    print(f"{primal_line}")
    print(f"{dual_line}")


with open(summary_file, "w") as f:
    header = (
        f"{'Graph':>5} | {'Verts':>5} | {'Edges(G)':>8} | "
        f"{'Edges(H)':>8} | {'LP':>10} | {'Root':>5} | "
        f"{'NonInt':>7} | {'Primal':>3} | {'Dual':>3} | {'Gap':>3}\n"
    )
    f.write(header)
    f.write("-" * len(header) + "\n")

    for _, s in results:
        f.write(
            f"{s['index']:>5} | "
            f"{s['vertices_G']:>5} | "
            f"{s['edges_G']:>8} | "
            f"{str(s['edges_H']) if s['edges_H'] is not None else 'NA':>8} | "
            f"{s['lp_status']:<10} | "
            f"{'Yes' if s['has_root'] else 'No':>5} | "
            f"{'Yes' if s['is_noninteger'] else 'No':>7} | "
            f"{fmt(s['primal'], 8)} | "
            f"{fmt(s['dual'],   8)} | "
            f"{fmt(s['gap'],    8)}\n"
        )

print(f"📝 Summary written to {summary_file}")

with PdfPages(output_pdf) as pdf:
    for i, (p, s) in enumerate(results):
        fig = p.matplotlib(figsize=(10, 10))
        pdf.savefig(fig)
        plt.close(fig)
print(f"\n✅ Combined {len(results)} graphs into {output_pdf}")


if mismatch_results:
    with PdfPages(comparison_pdf) as pdf:
        for comp, s in mismatch_results:

            fig = comp.matplotlib(figsize=(12, 4))

            plt.suptitle(
                f"Graph {s['index']} — "
                f"LP={s['lp_status']} | "
                f"Root={s['has_root']} | "
                f"{'Non-integer LP' if s['is_noninteger'] else ''}"
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"⚠️ Comparison mismatches saved to {comparison_pdf}")

✅ Graph    0 | LP=feasible   | Root=True  | Primal=   0.000 | Dual=   0.000 | Gap=   0.000
All0
70nz [beta:21, gamma:21, theta:21, iota:7]
✅ Graph    1 | LP=feasible   | Root=True  | Primal=  -2.000 | Dual=  -2.000 | Gap=   0.000
6nz [x:2, y:2, z:2]
71nz [alpha:1, beta:20, gamma:21, eta:1, theta:21, iota:7]
✅ Graph    2 | LP=infeasible | Root=False | Primal=      NA | Dual=      NA | Gap=      NA
None
None
✅ Graph    3 | LP=infeasible | Root=False | Primal=      NA | Dual=      NA | Gap=      NA
None
None
✅ Graph    4 | LP=infeasible | Root=False | Primal=      NA | Dual=      NA | Gap=      NA
None
None
✅ Graph    5 | LP=infeasible | Root=False | Primal=      NA | Dual=      NA | Gap=      NA
None
None
✅ Graph    6 | LP=infeasible | Root=False | Primal=      NA | Dual=      NA | Gap=      NA
None
None
✅ Graph    7 | LP=feasible   | Root=True  | Primal=  -4.000 | Dual=  -4.000 | Gap=   0.000
12nz [x:4, y:4, z:4]
78nz [alpha:2, beta:19, gamma:20, delta:1, epsilon:6, eta:1, theta:21, iot

<Figure size 640x480 with 0 Axes>

In [20]:
# Analyze graph summary with primal + dual classification
n = 7
filename = f"dual_graph_output/n={n}/graph_summary_n={n}.txt"

from collections import defaultdict

summary = defaultdict(list)

violations = []

with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line or line.startswith("-") or line.startswith("Graph"):
            continue

        parts = [p.strip() for p in line.split("|")]

        if len(parts) != 10:
            continue

        graph, verts, edges_g, edges_h, lp, root, nonint, primal, dual, gap = parts

        H_exists = (root == "Yes")

        if lp == "infeasible":
            primal_type = "Primal infeasible"
        elif nonint == "Yes":
            primal_type = "Primal fractional"
        else:
            primal_type = "Primal integer"

        if dual == "NA":
            dual_type = "Dual infeasible"
        else:
            try:
                gap_val = float(gap)

                if abs(gap_val) < 1e-6:
                    dual_type = "Dual feasible"
                elif gap_val > 0:
                    dual_type = "Dual unbounded"
                else:
                    dual_type = "Dual infeasible"

            except ValueError:
                dual_type = "Dual infeasible"

        if lp == "infeasible" and root != "No":
            violations.append(parts)

        key = (H_exists, primal_type, dual_type)
        summary[key].append(graph)

for H_exists in [True, False]:
    print(f"H exists: {H_exists}")
    print("-" * 50)

    for primal_type in [
        "Primal integer",
        "Primal fractional",
        "Primal infeasible",
    ]:
        for dual_type in [
            "Dual feasible",
            "Dual infeasible",
            "Dual unbounded",
        ]:
            key = (H_exists, primal_type, dual_type)
            graphs = summary.get(key, [])

            if graphs:
                print(
                    f"{primal_type:<20} | "
                    f"{dual_type:<17} → "
                    f"Count: {len(graphs):<3} | "
                    f"Graphs: {', '.join(graphs)}"
                )

    print()

if violations:
    print("⚠️ Logical violations detected:")
    for v in violations:
        print(v)
else:
    print("✅ No LP infeasible / Root inconsistency detected.")

H exists: True
--------------------------------------------------
Primal integer       | Dual feasible     → Count: 37  | Graphs: 0, 1, 7, 10, 25, 57, 61, 86, 101, 119, 347, 369, 375, 380, 412, 417, 521, 528, 543, 648, 655, 666, 830, 916, 934, 937, 938, 940, 946, 948, 952, 982, 1006, 1023, 1026, 1028, 1043
Primal fractional    | Dual feasible     → Count: 40  | Graphs: 159, 262, 276, 334, 425, 445, 615, 640, 676, 712, 763, 766, 772, 779, 781, 831, 871, 877, 896, 897, 903, 914, 954, 980, 988, 992, 1010, 1012, 1013, 1015, 1017, 1022, 1030, 1031, 1032, 1038, 1039, 1040, 1041, 1042

H exists: False
--------------------------------------------------
Primal fractional    | Dual feasible     → Count: 200 | Graphs: 42, 52, 56, 69, 115, 140, 150, 155, 173, 177, 181, 187, 210, 227, 239, 259, 273, 287, 291, 295, 303, 304, 306, 313, 326, 332, 338, 339, 340, 344, 345, 351, 353, 374, 383, 388, 404, 427, 430, 432, 439, 442, 443, 448, 471, 479, 490, 495, 499, 512, 516, 534, 536, 539, 541, 546, 553, 55